In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv('patients.csv')

# Keep a flag: which rows were originally missing (missingness can be informative)
df['DiagnosisImputed'] = df['Diagnosis'].isna().astype(int)

known   = df[df['Diagnosis'].notna()]
unknown = df[df['Diagnosis'].isna()]

features = ['Age', 'LabResult']
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(known[features], known['Diagnosis'])

df.loc[df['Diagnosis'].isna(), 'Diagnosis'] = clf.predict(unknown[features])

print(df['Diagnosis'].value_counts())
print(f"\nImputed {df['DiagnosisImputed'].sum()} rows")

Diagnosis
Diabetes        33
Flu             29
Hypertension    20
Common Cold     18
Name: count, dtype: int64

Imputed 21 rows


In [2]:
import pandas as pd

# ----- 1. Load -----
customers = pd.read_csv('customers.csv')
products  = pd.read_csv('products.csv')
orders    = pd.read_csv('orders.csv', parse_dates=['order_date'])
ratings = pd.read_csv('ratings.csv')

TOP_N = 5

# ----- 2. Enriched order-line table: attach price/name, compute revenue -----
lines = orders.merge(products, on='product_id', how='left')
lines['revenue'] = lines['price'] * lines['quantity']      # price is per-unit

# ----- 3. Top products by REVENUE -----
top_by_revenue = (lines.groupby(['product_id', 'product_name'], as_index=False)
                       .agg(revenue=('revenue', 'sum'), units=('quantity', 'sum'))
                       .sort_values('revenue', ascending=False)
                       .head(TOP_N))

# ----- 4. Top products by UNITS sold -----
top_by_units = (lines.groupby(['product_id', 'product_name'], as_index=False)
                     .agg(units=('quantity', 'sum'), revenue=('revenue', 'sum'))
                     .sort_values('units', ascending=False)
                     .head(TOP_N))

# ----- 5. Top clients for the LAST MONTH -----
# "Last month" = most recent calendar month present in the data (robust to stale data)
latest = lines['order_date'].max()
last_month = latest.to_period('M')
last_month_lines = lines[lines['order_date'].dt.to_period('M') == last_month]

top_clients = (last_month_lines.groupby('customer_id', as_index=False)
                               .agg(revenue=('revenue', 'sum'),
                                    units=('quantity', 'sum'),
                                    orders=('order_id', 'nunique'))
                               .merge(customers, on='customer_id', how='left')
                               .sort_values('revenue', ascending=False)
                               .head(TOP_N))

# ----- Report -----
pd.options.display.float_format = '{:,.2f}'.format
print(f'=== Top {TOP_N} products by REVENUE ===')
print(top_by_revenue.to_string(index=False))
print(f'\n=== Top {TOP_N} products by UNITS sold ===')
print(top_by_units.to_string(index=False))
print(f'\n=== Top {TOP_N} clients for {last_month} (latest month in data) ===')
print(top_clients[['customer_id', 'name', 'revenue', 'units', 'orders']].to_string(index=False))

=== Top 5 products by REVENUE ===
 product_id        product_name  revenue  units
          6 Brimnes Bed Storage    59521     77
         42    Småstad Wardrobe    51688     52
         29        Råskog Stool    50445     59
         26       Docksta Table    46386     54
         31        Nockeby Sofa    45646     58

=== Top 5 products by UNITS sold ===
 product_id            product_name  units  revenue
         39   Mackapar Shoe Storage     80    36640
          6     Brimnes Bed Storage     77    59521
         49  Söderhamn Sofa Section     63    27279
          2          Poäng Armchair     63    35847
         38 Bekant Conference Table     62    27342

=== Top 5 clients for 2023-09 (latest month in data) ===
 customer_id        name  revenue  units  orders
          44 Customer_44    13819     16       5
           1  Customer_1    12685     18       8
          33 Customer_33    11073     15       6
          18 Customer_18    11053     14       5
          52 Customer_52 

In [3]:
import pandas as pd

# ----- 1. Load -----
customers = pd.read_csv('customers.csv')
products  = pd.read_csv('products.csv')
orders    = pd.read_csv('orders.csv', parse_dates=['order_date'])

# ----- 2. Revenue per order line -----
lines = orders.merge(products[['product_id', 'price']], on='product_id', how='left')
lines['revenue'] = lines['price'] * lines['quantity']

# ----- 3. Snapshot date -----
# Anchor "today" to the day after the last order (data is historical).
# Swap in pd.Timestamp.today() if you want recency relative to the real calendar.
snapshot = lines['order_date'].max() + pd.Timedelta(days=1)

# ----- 4. Build R, F, M per customer -----
rfm = (lines.groupby('customer_id')
            .agg(recency=('order_date', lambda s: (snapshot - s.max()).days),  # days since last order
                 frequency=('order_id', 'nunique'),                            # distinct orders
                 monetary=('revenue', 'sum'))                                  # total spend
            .reset_index())

# ----- 5. Score each dimension 1-5 (quintiles) -----
# rank(method='first') guarantees unique ranks so qcut never chokes on tied edges.
# Recency: SMALLER is better -> ascending=False so the smallest recency lands in the top bin.
rfm['R'] = pd.qcut(rfm['recency'].rank(method='first', ascending=False), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RFM_score'] = rfm[['R', 'F', 'M']].sum(axis=1)

# ----- 6. Name the segments (action-oriented) -----
def segment(row):
    r, f = row['R'], row['F']
    if r >= 4 and f >= 4:   return 'Champions'          # recent + frequent = your best
    if f >= 4:              return 'Loyal'              # buy often, cooling slightly
    if r >= 4 and f >= 2:   return 'Potential Loyalist' # recent, building habit
    if r >= 4:              return 'New / Recent'       # just arrived, low frequency
    if r <= 2 and f >= 4:   return "Can't Lose"         # were great, now going quiet
    if r <= 2 and f >= 2:   return 'At Risk'            # slipping away
    if r <= 2:              return 'Hibernating / Lost' # long gone
    return 'Needs Attention'                            # mid-tier, nudge them

rfm['segment'] = rfm.apply(segment, axis=1)
rfm = rfm.merge(customers, on='customer_id', how='left')

# ----- 7. Report -----
pd.options.display.float_format = '{:,.2f}'.format
print(f'Snapshot date: {snapshot.date()}  |  customers: {len(rfm)}\n')

print('=== Segment summary (where your revenue lives) ===')
summary = (rfm.groupby('segment')
              .agg(customers=('customer_id', 'count'),
                   total_revenue=('monetary', 'sum'),
                   avg_revenue=('monetary', 'mean'),
                   avg_frequency=('frequency', 'mean'),
                   avg_recency_days=('recency', 'mean'))
              .sort_values('total_revenue', ascending=False))
print(summary.to_string())

print('\n=== Top 10 customers by RFM score ===')
cols = ['customer_id', 'name', 'recency', 'frequency', 'monetary', 'R', 'F', 'M', 'RFM_score', 'segment']
print(rfm.sort_values(['RFM_score', 'monetary'], ascending=False).head(10)[cols].to_string(index=False))

# rfm.to_csv('rfm_segments.csv', index=False)   # uncomment to save

Snapshot date: 2023-09-18  |  customers: 100

=== Segment summary (where your revenue lives) ===
                    customers  total_revenue  avg_revenue  avg_frequency  avg_recency_days
segment                                                                                   
Loyal                      23         386544    16,806.26          13.39              7.74
Champions                  17         320997    18,882.18          13.53              1.53
Potential Loyalist         19         235220    12,380.00           9.11              2.05
At Risk                    14         175321    12,522.93           9.00             12.50
Needs Attention            11         115399    10,490.82           7.27              5.27
Hibernating / Lost         12          89937     7,494.75           5.17             11.83
New / Recent                4          26749     6,687.25           5.25              3.00

=== Top 10 customers by RFM score ===
 customer_id        name  recency  frequency 

In [4]:
import pandas as pd

# ----- Load -----
products = pd.read_csv('products.csv')
ratings  = pd.read_csv('ratings.csv')

TOP_N = 10

# ----- 1. Reviews per product: how many, and how well rated -----
product_reviews = (ratings.groupby('product_id')
                          .agg(num_reviews=('rating', 'count'),
                               avg_rating=('rating', 'mean'))
                          .reset_index()
                          .merge(products[['product_id', 'product_name']], on='product_id', how='left'))

# Most-reviewed (by volume)
top_by_volume = product_reviews.sort_values('num_reviews', ascending=False).head(TOP_N)

# Best-rated (require a few reviews so a single 5-star doesn't win)
MIN_REVIEWS = 3
top_by_rating = (product_reviews[product_reviews['num_reviews'] >= MIN_REVIEWS]
                 .sort_values(['avg_rating', 'num_reviews'], ascending=False).head(TOP_N))

# ----- 2. Average number of reviews per customer -----
reviews_per_customer = ratings.groupby('customer_id').size()
avg_reviews = reviews_per_customer.mean()

# ----- Report -----
pd.options.display.float_format = '{:,.2f}'.format
cols = ['product_id', 'product_name', 'num_reviews', 'avg_rating']

print(f'=== Top {TOP_N} MOST-REVIEWED items ===')
print(top_by_volume[cols].to_string(index=False))

print(f'\n=== Top {TOP_N} BEST-RATED items (>= {MIN_REVIEWS} reviews) ===')
print(top_by_rating[cols].to_string(index=False))

print('\n=== Reviews per customer ===')
print(f'Total reviews            : {len(ratings)}')
print(f'Customers who reviewed   : {reviews_per_customer.size}')
print(f'Avg reviews / reviewer   : {avg_reviews:.2f}')
print(f'Median reviews / reviewer: {reviews_per_customer.median():.0f}')
print(f'Max reviews by one cust  : {reviews_per_customer.max()}')

=== Top 10 MOST-REVIEWED items ===
 product_id            product_name  num_reviews  avg_rating
         12          Raskog Trolley           16        2.31
         38 Bekant Conference Table           14        3.43
          7         Lack Side Table           13        3.00
         31            Nockeby Sofa           12        3.00
         45        Hektar Work Lamp           11        3.45
         32            Kivik Chaise           11        3.18
         28        Tarva Nightstand           11        2.55
         27            Ivar Cabinet           11        3.00
         17  Sinnerlig Pendant Lamp           11        2.91
         30    Strandmon Wing Chair           11        2.91

=== Top 10 BEST-RATED items (>= 3 reviews) ===
 product_id         product_name  num_reviews  avg_rating
         20    Pjatteryd Picture            6        4.17
          4       Malm Bed Frame           10        3.90
         44      Koppang Dresser            7        3.86
         14   

In [5]:
import pandas as pd

# ----- Load -----
customers = pd.read_csv('customers.csv')
products  = pd.read_csv('products.csv')
orders    = pd.read_csv('orders.csv', parse_dates=['order_date'])
ratings   = pd.read_csv('ratings.csv')

# ----- Per-customer aggregates -----
lines = orders.merge(products[['product_id', 'price']], on='product_id', how='left')
lines['revenue'] = lines['price'] * lines['quantity']

spend = (lines.groupby('customer_id')
              .agg(total_spent=('revenue', 'sum'),
                   last_purchase=('order_date', 'max'))
              .reset_index())

review_counts = (ratings.groupby('customer_id')
                        .size()
                        .reset_index(name='num_reviews'))

# ----- Attach to the customer table (left joins keep every customer) -----
customers = (customers
             .merge(spend, on='customer_id', how='left')
             .merge(review_counts, on='customer_id', how='left'))

# Customers with no orders/reviews -> 0 (last_purchase stays NaT = never purchased)
customers['total_spent'] = customers['total_spent'].fillna(0)
customers['num_reviews'] = customers['num_reviews'].fillna(0).astype(int)

# ----- Report -----
pd.options.display.float_format = '{:,.2f}'.format
print(customers.head(10).to_string(index=False))
print(f'\nRows: {len(customers)}  |  columns: {list(customers.columns)}')

# customers.to_csv('customers_enriched.csv', index=False)   # uncomment to save

 customer_id        name  total_spent              last_purchase  num_reviews
           1  Customer_1        27081 2023-09-17 08:39:23.971834            4
           2  Customer_2         9347 2023-09-15 08:39:23.971834            2
           3  Customer_3        18077 2023-09-13 08:39:23.971834            4
           4  Customer_4        25717 2023-09-14 08:39:23.971834            7
           5  Customer_5        14913 2023-09-09 08:39:23.971834            2
           6  Customer_6        13935 2023-09-17 08:39:23.971834            4
           7  Customer_7        16588 2023-09-17 08:39:23.971834            5
           8  Customer_8        18716 2023-09-17 08:39:23.971834            5
           9  Customer_9        19154 2023-09-02 08:39:23.971834            2
          10 Customer_10         7303 2023-09-10 08:39:23.971834            4

Rows: 100  |  columns: ['customer_id', 'name', 'total_spent', 'last_purchase', 'num_reviews']
